# Tableau de bord du robot

Le tableau de bord lit la batterie, permet de surveiller et de commander les moteurs à la main, puis lit le lidar et la caméra. **Chaque cellule est indépendante** : elle ouvre ce dont elle a besoin (moteurs, lidar, caméra) et le referme. Exécutez celles qui vous intéressent, dans l'ordre que vous voulez, avec **Maj + Entrée**. Le code s'exécute sur la Raspberry Pi du robot ; les résultats s'affichent sous chaque cellule.

**Sécurité** : les curseurs de la partie *Moteurs* font tourner une roue. Robot **sur sa cale**, roues en l'air.

## Batterie

Les moteurs Dynamixel sont branchés directement sur la batterie (3 éléments Li-ion en série) et mesurent sa tension. Un élément est plein vers 4,2 V ; en dessous de 3,3 V, il faut arrêter et recharger.

In [ ]:
import statistics

from holorobot.motors import voltages

voltage = statistics.median(voltages().values())  # tension lue par chaque moteur, sans rien leur changer
print(f"Batterie : {voltage:.1f} V, soit {voltage / 3:.2f} V par élément")

## Moteurs

Chaque cellule ouvre le bus des moteurs, fait son travail et le referme. Le bus ne sert qu'à un programme à la fois : tant qu'une roue tourne avec le curseur de vitesse, les autres cellules qui lisent les moteurs (batterie comprise) répondent que le bus est déjà utilisé. Documentation : `docs/moteurs.md` dans le dépôt.

### Les moteurs

In [ ]:
from holorobot.motors import status

state = status()  # l'état de chaque moteur, lu sans rien lui changer
if not state:
    print("Aucun moteur ne répond : les moteurs sont-ils alimentés (batterie) ?")
mode_name = {"wheel": "roue", "joint": "articulation"}
print(f"{'ID':>3}  {'modèle':8} {'mode':13} {'position':>9} {'tension':>8} {'température':>12}")
for i, motor in state.items():
    print(f"{i:>3}  {motor['model']:8} {mode_name[motor['mode']]:13} {motor['position']:8.1f}° "
          f"{motor['voltage']:6.1f} V {motor['temperature']:9.0f} °C")

### Températures

Un moteur qui force chauffe. Il se coupe de lui-même à 70 °C : laissez-le refroidir avant d'en arriver là. Relancez la cellule pour mettre à jour.

In [ ]:
import matplotlib.pyplot as plt
from holorobot.motors import status

temperatures = {i: motor["temperature"] for i, motor in status().items()}
fig, ax = plt.subplots(figsize=(6, 3))
bar_colors = ["tab:green" if t < 55 else "tab:orange" if t < 65 else "tab:red" for t in temperatures.values()]
ax.bar([f"moteur {i}" for i in temperatures], list(temperatures.values()), color=bar_colors)
ax.axhline(70, color="tab:red", ls="--")
ax.text(-0.4, 72, "coupure à 70 °C", color="tab:red")
ax.set_ylim(0, 80)
ax.set_ylabel("°C")
ax.set_title("Températures des moteurs")
plt.show()

### Changer l'identifiant d'un moteur

Sur le bus, chaque moteur doit avoir son propre identifiant : deux moteurs de même identifiant se brouillent. Un Dynamixel neuf sort d'usine avec l'identifiant 1. Branchez donc les moteurs neufs un par un, et donnez à chacun le sien avant de brancher le suivant.

Un moteur neuf communique aussi à 1 Mbit/s, alors que le bus du robot est réglé à 57 600 bauds : il n'apparaît pas tant qu'on n'a pas changé sa vitesse de communication. Demandez à l'enseignant.

In [ ]:
from holorobot.motors import Motors, find_ids

OLD_ID = None   # identifiant actuel, par exemple 1
NEW_ID = None   # nouvel identifiant, libre, par exemple 3

if OLD_ID is None or NEW_ID is None:
    print("Renseignez OLD_ID et NEW_ID, puis relancez la cellule.")
else:
    with Motors(ids=[OLD_ID], wheel_mode=False) as motors:  # n'ouvre que ce moteur, sans changer son mode
        motors.change_id(OLD_ID, NEW_ID)
    print(f"Le moteur {OLD_ID} répond maintenant à l'identifiant {NEW_ID}.")
print("Moteurs présents :", find_ids())

### Vitesse d'un moteur au curseur

Choisissez un moteur, appuyez sur **Marche**, puis réglez la vitesse en °/s. Tant que le bouton est enfoncé, une tâche de fond garde le moteur ouvert et lui renvoie la consigne 10 fois par seconde : le chien de garde arrête la roue si elle cesse d'arriver. Le moteur passe en mode roue s'il ne l'était pas. Sa vitesse mesurée et sa température s'affichent sous le curseur. **Arrêt** arrête la roue et libère le bus.

Pour arrêter, utilisez bien le bouton **Arrêt** : le menu *Interrupt Kernel* n'arrête pas cette tâche de fond.

In [ ]:
import threading
import time

import ipywidgets as widgets
from IPython.display import display
from holorobot.motors import Motors, find_ids

motor_ids = find_ids()
if not motor_ids:
    raise RuntimeError("Aucun moteur ne répond : les moteurs sont-ils alimentés (batterie) ?")
speed_motor = widgets.Dropdown(options=motor_ids, description="Moteur")
speed = widgets.IntSlider(value=0, min=-360, max=360, step=10, description="Vitesse (°/s)",
                          style={"description_width": "initial"}, layout=widgets.Layout(width="450px"))
speed_on = widgets.ToggleButton(value=False, description="Marche", icon="play")
speed_info = widgets.Label("")
speed_thread = None  # la tâche de fond qui renvoie la consigne


def run_speed_slider():
    """Ouvre le moteur choisi et lui renvoie la consigne du curseur toutes les 0,1 s, tant que le bouton
    est enfoncé ; à la fin, la roue s'arrête et le bus est libéré."""
    motor = speed_motor.value
    try:
        with Motors(ids=[motor]) as motors:  # passe le moteur en mode roue s'il ne l'était pas
            while speed_on.value and speed_motor.value == motor:
                motors.set_speeds({motor: speed.value})
                try:
                    measured, temperature = motors.get_speeds()[motor], motors.get_temperatures()[motor]
                    speed_info.value = f"vitesse mesurée : {measured:+.0f} °/s | température : {temperature:.0f} °C"
                except Exception:  # une mesure ratée sur le bus n'arrête pas la roue
                    pass
                time.sleep(0.1)
        speed_info.value = f"moteur {motor} arrêté, bus libéré"
    except Exception as error:
        speed_info.value = f"arrêt : {error}"
    finally:
        speed_on.value = False


def on_speed_toggle(change):
    global speed_thread
    speed_on.description, speed_on.icon = ("Arrêt", "stop") if change["new"] else ("Marche", "play")
    if change["new"] and (speed_thread is None or not speed_thread.is_alive()):
        speed_thread = threading.Thread(target=run_speed_slider, daemon=True)
        speed_thread.start()


speed_on.observe(on_speed_toggle, "value")
display(widgets.VBox([widgets.HBox([speed_motor, speed_on]), speed, speed_info]))

### Position d'un moteur au curseur

Choisissez un moteur et déplacez le curseur : une tâche de fond passe le moteur en mode articulation et l'amène à la position demandée (en degrés, de −180 à 180), à la vitesse choisie. Si vous rebougez le curseur pendant le trajet, le moteur repart vers la nouvelle position. Arrivé, il garde sa position, même une fois le bus libéré. **Revenir en mode roue** le remet en roue, à l'arrêt et sans couple.

In [ ]:
import threading

import ipywidgets as widgets
from IPython.display import display
from holorobot.motors import Motors, find_ids, status


def current_position(motor):
    """Position actuelle du moteur, en degrés arrondis, lue sans rien lui changer."""
    return round(status(motor)[motor]["position"])


motor_ids = find_ids()
if not motor_ids:
    raise RuntimeError("Aucun moteur ne répond : les moteurs sont-ils alimentés (batterie) ?")
position_motor = widgets.Dropdown(options=motor_ids, description="Moteur")
position = widgets.FloatSlider(value=current_position(position_motor.value), min=-180, max=180, step=1,
                               description="Position (°)", continuous_update=False,
                               style={"description_width": "initial"}, layout=widgets.Layout(width="450px"))
move_speed = widgets.IntSlider(value=90, min=10, max=360, step=10, description="Vitesse (°/s)",
                               style={"description_width": "initial"})
back_to_wheel = widgets.Button(description="Revenir en mode roue", icon="undo", layout=widgets.Layout(width="200px"))
position_info = widgets.Label("")
position_request = threading.Event()  # réveille la tâche de fond : le curseur a bougé, ou le bouton a été cliqué
wheel_wanted = False      # vrai quand le bouton « Revenir en mode roue » a été cliqué
following_motor = False   # vrai quand le curseur suit le moteur choisi : ce n'est pas une consigne


def follow_position_slider(request):
    """Tâche de fond : à chaque demande, ouvre le moteur choisi, l'amène à la position du curseur (la plus
    récente, si le curseur a rebougé entre-temps) ou le remet en mode roue, puis libère le bus."""
    global wheel_wanted
    while True:
        request.wait()
        request.clear()
        motor = position_motor.value
        try:
            if wheel_wanted:
                wheel_wanted = False
                with Motors(ids=[motor], wheel_mode=False) as motors:  # en sortant : couple coupé, bus libéré
                    motors.set_wheel_mode([motor])
                position_info.value = f"moteur {motor} en mode roue, à l'arrêt"
                continue
            motors = Motors(ids=[motor], wheel_mode=False)
            try:
                if motors.modes[motor] != "joint":
                    motors.set_joint_mode([motor], speed=move_speed.value)
                goal = None
                while position.value != goal and position_motor.value == motor:
                    request.clear()  # ce trajet répond à toutes les demandes reçues jusqu'ici
                    goal = position.value
                    position_info.value = f"moteur {motor} en route vers {goal:+.0f}°…"
                    current = motors.get_positions()[motor]
                    timeout = abs(goal - current) / move_speed.value + 2  # le temps du trajet, plus une marge
                    reached = motors.move_to({motor: goal}, speed=move_speed.value, timeout=timeout)[motor]
                    position_info.value = f"moteur {motor} arrivé à {reached:+.1f}° (consigne {goal:+.0f}°)"
            finally:
                motors.close(hold=True)  # libère le bus ; le moteur garde sa consigne et s'y tient
        except Exception as error:
            position_info.value = f"erreur : {error}"


def on_position_motor(change):
    global following_motor
    following_motor = True
    try:
        position.value = current_position(change["new"])  # le curseur part de la position actuelle
    except Exception as error:
        position_info.value = f"erreur : {error}"
    finally:
        following_motor = False


def on_position(change):
    if not following_motor:
        position_info.value = f"moteur {position_motor.value} : demandé {change['new']:+.0f}°…"
        position_request.set()


def on_back_to_wheel(_):
    global wheel_wanted
    wheel_wanted = True
    position_request.set()


threading.Thread(target=follow_position_slider, args=(position_request,), daemon=True).start()
position_motor.observe(on_position_motor, "value")
position.observe(on_position, "value")
back_to_wheel.on_click(on_back_to_wheel)
display(widgets.VBox([widgets.HBox([position_motor, back_to_wheel]), position, move_speed, position_info]))

### Relâcher les moteurs

Arrête toutes les roues et coupe le couple de tous les moteurs, sans changer leur mode : ils tournent alors librement à la main. Utile après le curseur de position, qui laisse le moteur tenir sa position.

In [ ]:
from holorobot.motors import Motors

with Motors(wheel_mode=False) as motors:
    motors.release()
print("Moteurs relâchés :", motors.ids)

## Lidar

Le lidar tourne sur lui-même environ 7 fois par seconde et mesure la distance des obstacles tout autour du robot. `get_scan()` renvoie un tour complet : des angles en degrés (sens des aiguilles d'une montre) et des distances en mètres. Documentation : `docs/lidar.md` dans le dépôt.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from holorobot.lidar import Lidar

with Lidar() as lidar:  # « with » arrête le moteur du lidar à la fin, même en cas d'erreur
    scan = lidar.get_scan()

angle, distance = scan.nearest()
print(f"{len(scan)} points ; obstacle le plus proche : {distance:.2f} m à {angle:.0f}°")

ax = plt.figure(figsize=(6, 6)).add_subplot(projection="polar")
ax.set_theta_zero_location("N")  # 0° en haut
ax.set_theta_direction(-1)       # sens des aiguilles d'une montre, comme le lidar
ax.scatter(np.radians(scan.angles), scan.distances, s=3)
ax.set_title("Scan du lidar (distances en m)")
plt.show()

### Segments de droite

La cellule prend un tour du lidar et y cherche les segments de droite avec `scan.segments()` (murs, faces de meubles), du plus sûr au moins sûr. Pour chacun : sa longueur, sa distance au lidar, la direction où il est vu, l'équation $a\,x + b\,y + c = 0$ de sa droite (repère du lidar : x vers son 0°, y vers sa gauche) et une confiance entre 0 et 1.

In [ ]:
import matplotlib.pyplot as plt
from holorobot.lidar import Lidar

with Lidar() as lidar:
    scan = lidar.get_scan()

segments = scan.segments()
print(f"{len(segments)} segments de droite ; les plus sûrs :")
for s in segments[:8]:
    a, b, c = s.equation
    print(f"  confiance {s.confidence:.2f} | {s.length:.2f} m, à {s.distance:.2f} m vu à {s.bearing:3.0f}° | "
          f"{a:+.2f}·x {b:+.2f}·y {c:+.2f} = 0")

# vue de dessus, le 0° du lidar en haut : un point (x, y) du lidar est dessiné en (-y, x)
xy = scan.xy()
fig, ax = plt.subplots(figsize=(7, 7))
ax.plot(-xy[:, 1], xy[:, 0], ".", color="0.6", ms=3)
colors = plt.get_cmap("viridis")
for s in segments:
    (x1, y1), (x2, y2) = s.start, s.end
    ax.plot([-y1, -y2], [x1, x2], color=colors(s.confidence), lw=4)
ax.plot(0, 0, "r^", ms=10)
ax.set_aspect("equal")
ax.grid(alpha=0.3)
ax.set_title("Segments de droite (couleur = confiance)")
fig.colorbar(plt.cm.ScalarMappable(cmap=colors, norm=plt.Normalize(0, 1)), ax=ax, shrink=0.75, label="confiance")
plt.show()

## Caméra

La caméra RealSense donne une image couleur et une carte de profondeur : pour chaque pixel, la distance en mètres. Les pixels sans mesure (trop près, à moins de 30 cm environ, ou surface mal vue) apparaissent en blanc.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pyrealsense2 as rs

UPSIDE_DOWN = True  # sur MobileRobot-1, la caméra est montée tête en bas : adaptez à votre robot

pipeline = rs.pipeline()
config = rs.config()
config.enable_stream(rs.stream.color, 640, 480, rs.format.rgb8, 30)
config.enable_stream(rs.stream.depth, 640, 480, rs.format.z16, 30)
profile = pipeline.start(config)
try:
    scale = profile.get_device().first_depth_sensor().get_depth_scale()  # mètres par unité
    align = rs.align(rs.stream.color)  # met la profondeur dans le même cadrage que la couleur
    for _ in range(30):  # laisse l'exposition automatique se régler
        pipeline.wait_for_frames()
    frames = align.process(pipeline.wait_for_frames())
    color = np.asanyarray(frames.get_color_frame().get_data()).copy()
    depth = np.asanyarray(frames.get_depth_frame().get_data()) * scale
finally:
    pipeline.stop()  # libère la caméra
if UPSIDE_DOWN:
    color, depth = color[::-1, ::-1], depth[::-1, ::-1]

print(f"distance au centre de l'image : {depth[240, 320]:.2f} m")
fig, (left, right) = plt.subplots(1, 2, figsize=(12, 4.5))
left.imshow(color)
left.set_title("Couleur")
image = right.imshow(np.ma.masked_equal(depth, 0), cmap="turbo", vmin=0.2, vmax=4)
right.set_title("Profondeur (m)")
for ax in (left, right):
    ax.axis("off")
fig.colorbar(image, ax=right)
plt.show()

### Objets reconnus par YOLO

La cellule prend une image avec la caméra, et YOLO y reconnaît des objets : il connaît 80 catégories (personne, chaise, table, bouteille, tasse, téléphone…). On utilise YOLO26 *nano*, le plus rapide : moins d'une seconde par image sur la Pi. La première exécution est plus longue : chargement de la bibliothèque, et téléchargement du modèle (5 Mo), gardé ensuite dans le dossier du carnet.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pyrealsense2 as rs
from ultralytics import YOLO

UPSIDE_DOWN = True  # sur MobileRobot-1, la caméra est montée tête en bas : adaptez à votre robot

pipeline = rs.pipeline()
config = rs.config()
config.enable_stream(rs.stream.color, 640, 480, rs.format.bgr8, 30)  # YOLO attend l'ordre bleu, vert, rouge
pipeline.start(config)
try:
    for _ in range(30):  # laisse l'exposition automatique se régler
        pipeline.wait_for_frames()
    bgr = np.asanyarray(pipeline.wait_for_frames().get_color_frame().get_data()).copy()
finally:
    pipeline.stop()  # libère la caméra
if UPSIDE_DOWN:
    bgr = np.ascontiguousarray(bgr[::-1, ::-1])

model = YOLO("yolo26n.pt")
result = model.predict(bgr, imgsz=640, conf=0.25, verbose=False)[0]
if len(result.boxes) == 0:
    print("Aucun objet reconnu : approchez une tasse, une bouteille, une chaise… et relancez la cellule.")
for box in result.boxes:
    print(f"{model.names[int(box.cls)]:15s} confiance {float(box.conf):.0%}")

plt.figure(figsize=(8, 6))
plt.imshow(result.plot()[..., ::-1])  # image annotée, remise en rouge, vert, bleu pour l'affichage
plt.axis("off")
plt.title("Objets reconnus par YOLO")
plt.show()

## Et ensuite ?

- Le carnet **Découverte des moteurs** (`decouverte_moteurs.ipynb`) explique la commande des roues, jusqu'à piloter le robot avec un vecteur vitesse et une rotation.
- Chaque capteur ne sert qu'à **un seul programme à la fois** : fermez-le après usage (`with`, `close()`, `pipeline.stop()`), ou redémarrez le noyau (menu *Kernel*).
- Documentation : dossier `docs/` du dépôt (lidar, moteurs).